# Matelda Pipeline

As data-driven applications gain popularity, ensuring high data quality is a growing concern. This requirement involves not only the quality of primary data sources but also external data sources used for data enrichment purposes. Yet, data cleaning techniques are limited to treating one table at a time. A table-by-table application of such methods is cumbersome, because these methods either require previous knowledge about constraints or often require labor-intensive configurations and manual labeling for each individual table. As a result, they hardly scale beyond a few tables and miss the chance for optimizing the cleaning process. To tackle these issues, we introduce a novel semi-supervised error detection approach, Matelda, that organizes a given set of tables by folding their cells with regard to domain and quality similarity to facilitate user supervision. The idea is to identify groups of data cells across all tables that can benefit from the same user label. For this purpose, we identify a feature embedding that makes cell values comparable across many different tables. Experimental evaluations demonstrate that Matelda outperforms various configurations of existing single-table cleaning methodologies in cleaning multiple tables at a time, in particular when the ratio of labeling budget to number of tables is very low.

For more information about Matelda, we recommend you to read the [Paper](https://openproceedings.org/2025/conf/edbt/paper-98.pdf) and view the corresponding Code on [GitHub](https://github.com/LUH-DBS/Matelda).

# Initialization of Matelda

The initialization function in Matelda sets up the necessary configurations, directories, and variables, including paths for input data, output, logs, and results. It prepares the environment for the error detection process, ensuring all required directories exist and the configuration parameters are loaded correctly.

In [1]:
from pipeline_functions import init, domain_based_folding, quality_based_folding, loading_columns_grouping_results
import multiprocessing
import os 
import pandas as pd
import pickle
import ipywidgets as widgets
from IPython.display import display

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML



In [2]:
!conda install ipywidgets -y

Channels:
 - defaults
 - conda-forge
Platform: linux-64
Solving environment: done

# All requested packages already installed.



In [3]:
# Initialization of Matelda

# Adjust the example configuration as needed
configs = {
    "EXPERIMENTS": {
        "labeling_budget": 27700,
        "exp_name": "test_edbt",
        "n_cores": 128,
        "save_mediate_res_on_disk": 1,
        "final_result_df": False,
    },
    "DIRECTORIES": {
        "sandbox_dir": "datasets",
        "tables_dir": "Quintet",
        "output_dir": "output_quintet/output_quintet",
        "results_dir": "results",
        "logs_dir": "logs",
        "aggregated_lake_path": "aggregated_lake",
        "dirty_files_name": "dirty.csv",
        "clean_files_name": "clean.csv",
    },
    "TABLE_GROUPING": {
        "tg_enabled": 1,
        "tg_res_available": 0,
        "tg_method": "bert",
    },
    "COLUMN_GROUPING": {
        "cg_enabled": 1,
        "cg_res_available": 0,
        "min_num_labes_per_col_cluster": 2,
        "cg_clustering_alg": "hac",
    },
    "CELL_GROUPING": {
        "cell_feature_generator_enabled": 1,
        "cell_clustering_alg": "km",
        "cell_clustering_res_available": 0,
        "classification_mode": 1,
        "labels_per_cell_group": 1,
    },
    "RAHA": {
        "save_results": False,
        "strategy_filtering": False,
        "error_detection_algorithms": "OD, RVD, RVD_orig",
    }
}

execution = 0 
configs = init(configs, execution)

# Or load the configuratoin parameters directly form the config.ini file
# configs = init(execution)

# Domain-Based Cell Folding
Domain-based cell folding in Matelda organizes cells from different tables by their semantic similarities.

In [4]:
# Set up multiprocessing pool (if necessary)
n_cores = configs["n_cores"]
pool = multiprocessing.Pool(n_cores)

# Call domain_based_folding
table_grouping_dict, table_size_dict = domain_based_folding(configs, pool)
# print(table_grouping_dict)

I need at least 2 labeled cells per table group to work at all and at least 2 * 6 labeled cells per table group to work effectively! Thant means you need to label 60 cells if you want reasonable (!) results:


### Presenting results from Domain-Based Cell Folding to user

In [5]:
def create_table_content_widget(csv_file, base_csv_dir, clean_file_name):
    """
    Create an interactive output widget for previewing and toggling the display of a CSV table.

    Additionally, the widget includes interactive controls:
      - A "Show complete table" button that, when clicked, replaces the preview with the full table display.
      - A "Minimize Table" button that appears after the full table is displayed, allowing the view to revert 
        back to showing only the preview.

    Parameters:
        csv_file (str): The corresponding table folder is derived by removing the extension.
        base_csv_dir (str): The base directory that contains the table folders.
        clean_file_name (str): The name of the CSV file within each table folder to be displayed (typically "clean.csv" (or "dirty.csv")).

    Returns:
        tuple: A tuple containing:
            - output_widget (ipywidgets.Output): The widget container for displaying the table content.
            - load_content_func (callable): A function that loads and renders the table preview and interactive controls.
    """

    table_name = os.path.splitext(csv_file)[0]
    output = widgets.Output()
    # Flag so that we load content only once initially
    output._loaded = False  

    def load_content():
        with output:
            if output._loaded:
                # Prevent reloading if already loaded
                return  
            output._loaded = True
            clear_output(wait=True)
            full_path = os.path.join(base_csv_dir, table_name, clean_file_name)
            try:
                df = pd.read_csv(full_path)
                
                # Function to display the head and "Show complete table" button.
                def display_head():
                    clear_output(wait=True)
                    display(HTML(f'<div style="overflow-x: auto; width:100%;">{df.head().to_html()}</div>'))
                    display(btn_show_full)
                
                # Callback for the "Show complete table" button.
                def on_show_full_table(b):
                    with output:
                        clear_output(wait=True)
                        display(HTML(f'<div style="overflow-x: auto; width:100%;">{df.to_html()}</div>'))
                        display(btn_minimize)
                
                # Callback for the "Minimize Table" button.
                def on_minimize_table(b):
                    with output:
                        display_head()
                
                # Create the buttons.
                btn_show_full = widgets.Button(description="Show complete table")
                btn_minimize = widgets.Button(description="Minimize Table")
                btn_show_full.on_click(on_show_full_table)
                btn_minimize.on_click(on_minimize_table)
                
                # Initially, display the head view.
                display_head()
                
            except Exception as e:
                print(f"Error loading {table_name} from {full_path}: {e}")

    return output, load_content


In [6]:
# ==================== #
# Configure File Paths #
# ==================== #

# Build the path to the pickle file dynamically using the configured experiment output path.
pickle_file = os.path.join(".", configs["experiment_output_path"], "table_group_dict.pickle")
# print("Pickle file path:", pickle_file)

# Build the base directory where the CSV files are stored (e.g., "./datasets/Quintet").
base_csv_dir = os.path.join(
    ".", 
    configs["configs"]["DIRECTORIES"]["sandbox_dir"], 
    configs["configs"]["DIRECTORIES"]["tables_dir"]
)
# print("Base CSV directory:", base_csv_dir)

# Retrieve the name of the clean CSV file.
clean_file_name = configs["configs"]["DIRECTORIES"]["clean_files_name"]

# =============================================================================
# Load Cell Folds from the Pickle File
# =============================================================================
with open(pickle_file, "rb") as file:
    data = pickle.load(file)
# Example data structure (for instance):
# {0: ['rayyan.csv', 'hospital.csv', 'movies_1.csv'],
#  1: ['beers2.csv', 'beers.csv'],
#  2: ['flights.csv']}
# print("Cell folds data:", data)

# =============================================================================
# Create Outer Accordion for Cell Folds (with one inner accordion per fold)
# =============================================================================

fold_keys = sorted(data.keys())
fold_widgets = []

for key in fold_keys:
    # List of CSV files for this cell fold
    csv_files = data[key] 
    # Will hold the Output widgets for each table 
    table_widgets = []    
    # Their corresponding load functions
    load_functions = []   
    
    # Create one widget (and load function) per table
    for csv_file in csv_files:
        widget_content, load_func = create_table_content_widget(csv_file, base_csv_dir, clean_file_name)
        table_widgets.append(widget_content)
        load_functions.append(load_func)
    
    # Create one inner accordion for all tables in this cell fold
    inner_accordion = widgets.Accordion(children=table_widgets)
    # Set the title of each section to the table name (without ".csv")
    for idx, csv_file in enumerate(csv_files):
        table_name = os.path.splitext(csv_file)[0]
        inner_accordion.set_title(idx, table_name)
    
    # Attach an observer so that when a section is expanded, its content loads
    def on_inner_change(change, load_funcs=load_functions, acc=inner_accordion):
        new_index = change.get("new", None)
        if new_index is not None and new_index < len(load_funcs):
            # Load content for the expanded section
            load_funcs[new_index]()  
    
    inner_accordion.observe(on_inner_change, names="selected_index")
    
    fold_widgets.append(inner_accordion)

# Create an outer accordion where each cell fold is a section
outer_accordion = widgets.Accordion(children=fold_widgets)
for idx, key in enumerate(fold_keys):
    outer_accordion.set_title(idx, f"Cell Fold {key}")

# =============================================================================
# Display the Outer Accordion
# =============================================================================
display(outer_accordion)

Pickle file path: ./output_quintet/output_quintet_0/_test_edbt_Quintet_27700_labels/table_group_dict.pickle
Cell folds data: {0: ['flights.csv'], 1: ['rayyan.csv'], 2: ['hospital.csv'], 3: ['beers.csv'], 4: ['movies_1.csv']}


Accordion(children=(Accordion(children=(Output(),), titles=('flights',)), Accordion(children=(Output(),), titl…

# Quality-Based Cell Folding

In [ ]:
# Use the keys returned by init (flattened names, not nested dictionaries)
cg_enabled = configs["column_grouping_enabled"]
column_groups_dir = os.path.join(configs["mediate_files_path"], "col_grouping_res", "col_df_res")
expected_file = os.path.join(column_groups_dir, "col_df_labels_cluster_0.pickle")

if cg_enabled and os.path.exists(expected_file):
    # Load the column grouping results to get the cluster sizes and the path to the column groups file.
    # This function returns a tuple of (number_of_col_clusters, cluster_sizes_dict, column_groups_df_path)
    _, cluster_sizes_dict, column_groups_df_path = loading_columns_grouping_results(
        table_grouping_dict, configs["mediate_files_path"]
    )

# Assuming column_groups_df_path and cluster_sizes_dict have been set by your domain-based folding cell:
if column_groups_df_path is not None:
    results = quality_based_folding(configs, pool, column_groups_df_path, cluster_sizes_dict)
    
    # Unpack the results
    (
        y_test_all, 
        y_local_cell_ids, 
        predicted_all, 
        y_labeled_by_user_all,
        unique_cells_local_index_collection, 
        samples, 
        n_user_labeled_cells
    ) = results
    
    # Present the results in the notebook
    print("Number of user labeled cells (quality based folding):", n_user_labeled_cells)
else:
    print("Skipping quality based folding due to missing column grouping results.")


### Presenting results from Quality-Based Cell Folding to user

In [ ]:
###############
# Problem:    #
###############
# - we discussed last time that I should look into the pickle files which are generated in error_detection
# - not sure what to visualize and what each column is
###############

# Load the DataFrame from your pickle file
df = pd.read_pickle('/home/julian/projects/Matelda/output_quintet/output_quintet_0/_test_edbt_Quintet_27700_labels/cell_clustering/all_cell_clusters_records.pickle')
#df = pd.read_pickle('/home/julian/projects/Matelda/output_quintet/output_quintet_0/_test_edbt_Quintet_27700_labels/cell_clustering/cell_cluster_cells_dict_all.pickle')
#columns_to_show = ['table_cluster', 'col_cluster', 'n_cells', 'cells_per_cluster']
#df_sample = df[columns_to_show].head(5)
df_sample = df.head(5)
df_sample

In [ ]:
####################
# Test             #
####################
# - tested the ipywidgets expand/collapse buttons
# - but thats probably not what we want to visualize
####################

# Load the DataFrame from your pickle file.
df = pd.read_pickle('/home/julian/projects/Matelda/output_quintet/output_quintet_0/_test_edbt_Quintet_27700_labels/cell_clustering/all_cell_clusters_records.pickle')

# Choose the columns you want to display and take a small sample.
columns_to_show = ['table_cluster', 'col_cluster', 'n_cells', 'cells_per_cluster']
df_sample = df[columns_to_show].head(5).copy()  # Using head(5) to keep it light.

def format_cells_per_cluster(cell_cluster):
    """
    Convert the cells_per_cluster dictionary into a multi-line string.
    For each key, show only the first 10 items of the list (if the list is long).
    """
    if isinstance(cell_cluster, dict):
        lines = []
        for key, value in cell_cluster.items():
            # Show a preview of the list; adjust the number (10 here) as needed.
            if isinstance(value, list) and len(value) > 10:
                preview = value[:10]
                lines.append(f"{key}: {preview} ... ({len(value)} items)")
            else:
                lines.append(f"{key}: {value}")
        return "\n".join(lines)
    return str(cell_cluster)

# Create an accordion widget for each row in the sample.
accordion_items = []
for idx, row in df_sample.iterrows():
    # Build a summary header for the accordion.
    header = f"Cluster: ({row['table_cluster']}, {row['col_cluster']}), n_cells: {row['n_cells']}"
    
    # Format the full details of cells_per_cluster.
    details = format_cells_per_cluster(row['cells_per_cluster'])
    
    # Create an output widget to hold the formatted details (using <pre> to preserve formatting).
    detail_output = widgets.HTML(value=f"<pre>{details}</pre>")
    
    # Create an Accordion widget with this detail.
    accordion = widgets.Accordion(children=[detail_output])
    accordion.set_title(0, header)
    
    accordion_items.append(accordion)

# Display all the accordions vertically.
display(widgets.VBox(accordion_items))


In [ ]:
samples